In [ ]:
import polars as pl

daily = pl.scan_parquet("../data/1d/1d.parquet")

In [ ]:
daily.head().collect(engine="streaming")

Let's check the average return of a volume-rocketing event.

In [ ]:
daily.with_columns(pl.col("close_time").cast(pl.Datetime)).with_columns(
    pl.col("close").pct_change().over("symbol").alias("return")
).group_by("close_time").agg(pl.col("return").mean()).sort("close_time").with_columns(
    pl.col("return").cum_sum().alias("cumulative_return")).collect(engine="streaming").plot.line(x="close_time", y="cumulative_return")

In [ ]:
data = daily.sort("close_time").with_columns(
    (pl.col("close").pct_change().shift(-1)).over("symbol").alias("return")
)
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).select(
    pl.col("return"), pl.col("close_time")
).group_by("close_time").agg(pl.col("return").mean()).with_columns(
    pl.col("return").cum_sum().alias("cum_return")
).sort("close_time").collect(engine="streaming").plot.line(
    x="close_time", y="cum_return"
)

In [ ]:
data = daily.with_columns(pl.col("close").pct_change().over("symbol").alias("return"))
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).count().collect(engine="streaming")

In [ ]:
data.select(pl.col("close_time")).unique().count().collect(engine="streaming")

In [ ]:
data = daily.with_columns(pl.col("close").pct_change().over("symbol").alias("return"))
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).select(pl.col("close_time")).unique().count().collect(engine="streaming")